In [1]:
# 1: setup + connect to ES
# purpose: connect and define index name.

from elasticsearch import Elasticsearch
import os
import numpy as np
import json

es = Elasticsearch("http://localhost:9200")

# is elasticsearch alive
print("Connected:", es.ping())

INDEX_NAME = "java-tfidf5"   # change this 9 times 
print(f"Index Name: {INDEX_NAME}")

Connected: True
Index Name: java-tfidf5


javascript-bm25
java-tfidf5

java-dirichlet5

python-bm25
python-tfidf5
python-dirichlet5

javascript-bm25
javascript-tfidf5
javascript-dirichlet5

In [ ]:
# 2. generate ratings (Algorithm 2)----------------------------------------------------------------------
# purpose: convert cosidf.txt into ratings dictionary. implement algorithm 2 from hw2part3

def load_ratings(cosidf_file, index_name):
    ratings_dict = {}

    with open(cosidf_file, "r") as f:
        next(f)  # skip header
        for line in f:
            qid1, qid2, score, label = line.strip().split("\t")
            ratings_dict.setdefault(qid1, []).append({
                "_index": index_name,
                "_id": qid2,
                "rating": int(label)
            })

    return ratings_dict


In [3]:
#3. ranking function ( figure 2)--------------------------------------------------------------

def ranking(qid, qid_title, ratings):

    search_body = {
        "requests": [
            {
                "id": str(qid),
                "request": {
                    "query": {
                        "bool": {
                            "must_not": {
                                "match": {"_id": qid}
                            },
                            "should": [
                                {
                                    "match": {
                                        "title": {
                                            "query": qid_title,
                                            "boost": 3.0
                                        }
                                    }
                                },
                                {
                                    "match": {
                                        "body": {
                                            "query": qid_title,
                                            "boost": 0.5
                                        }
                                    }
                                },
                                {
                                    "match": {
                                        "answer": {
                                            "query": qid_title,
                                            "boost": 0.5
                                        }
                                    }
                                }
                            ]
                        }
                    }
                },
                "ratings": ratings
            }
        ],
        "metric": {
            "dcg": {
                "k": 10,
                "normalize": True
            }
        }
    }

    return search_body

In [4]:
#4 — evaluation loop (hw2part3: algorithm 1)-----------------------------------------------------------------------
#purpose: compute NDCG@10 for all qid1.

dataset = INDEX_NAME.split("-")[0]
cosidf_file = f"{dataset}/{dataset}_cosidf.txt"
ratings_dict = load_ratings(cosidf_file, INDEX_NAME)

print(f"using ratings file: {cosidf_file}")
print(f"Index Name: {INDEX_NAME}")


using ratings file: java/java_cosidf.txt
Index Name: java-tfidf5


In [5]:
# #Rank_Eval v1
# # Loop through each query ID, use its title as the search query, and run Elasticsearch rank_eval.
# # Compute and store the NDCG@10 score for each query based on the provided relevance ratings.

# ndcg_scores = []


# for qid1 in ratings_dict.keys():

#     # get title of qid1 from index
#     doc = es.get(index=INDEX_NAME, id=qid1, _source=["title"])
#     qid1_title = doc["_source"]["title"]

#     ratings = ratings_dict[qid1]

#     search_body = ranking(qid1, qid1_title, ratings)

#     result = es.rank_eval(index=INDEX_NAME, body=search_body)

#     ndcg = result["metric_score"]
#     ndcg_scores.append(ndcg)

#     print(f"{qid1}: {ndcg}")

In [6]:
#5. RANK-EVAL v2-----------------------------------------------------------------------
# # loop through each query ID, use its title as the search query, and run Elasticsearch rank_eval.
# compute and store the NDCG@10 score for each query based on the provided relevance ratings.

ndcg_scores = []

# #V2.2: fetch qid and ratings together --> MIGHT MAKE IT FASTER

#for qid1 in ratings_dict.keys():
for qid1, ratings in ratings_dict.items():

    # get title of qid1 from index
    doc = es.get(index=INDEX_NAME, id=qid1, _source=["title"])
    qid1_title = doc["_source"]["title"]

    #ratings = ratings_dict[qid1] --> went to beginnign of loop

    search_body = ranking(qid1, qid1_title, ratings)

    result = es.rank_eval(index=INDEX_NAME, body=search_body)

    ndcg = result["metric_score"]
    ndcg_scores.append(ndcg)

   # print(f"{qid1}: {ndcg}")
   #V2.1: PRINTING LESS OFTEN --> MIGHT MAKE IT FASTER
    if len(ndcg_scores) % 50 == 0:
        print("processed", len(ndcg_scores))

processed 50
processed 100
processed 150
processed 200
processed 250
processed 300
processed 350
processed 400
processed 450
processed 500
processed 550
processed 600
processed 650
processed 700
processed 750
processed 800
processed 850
processed 900
processed 950
processed 1000
processed 1050
processed 1100
processed 1150
processed 1200
processed 1250
processed 1300
processed 1350
processed 1400
processed 1450
processed 1500
processed 1550
processed 1600
processed 1650
processed 1700
processed 1750
processed 1800
processed 1850
processed 1900
processed 1950
processed 2000
processed 2050
processed 2100
processed 2150
processed 2200
processed 2250
processed 2300
processed 2350
processed 2400
processed 2450
processed 2500
processed 2550
processed 2600
processed 2650
processed 2700
processed 2750
processed 2800
processed 2850
processed 2900
processed 2950
processed 3000
processed 3050
processed 3100
processed 3150
processed 3200
processed 3250
processed 3300
processed 3350
processed 3400


- processed 8250
- processed 8300
- processed 8350
- processed 8400

java
- print(len(ratings_dict))
- 8448
  
- python
- javacript
  

- avg time to run: 4 min 30 sec

In [7]:
# 5. FINAL NDCG@10 SCORE-----------------------------------------------------------------------
# Purpose: compute final average NDCG@10

average_ndcg = np.mean(ndcg_scores)

print(f"{INDEX_NAME} ndcg@10: {average_ndcg}")

java-tfidf5 ndcg@10: 0.4598704998596104


List of all results
-------------------------
- java-bm25 ndcg@10: 0.47646329314588537
- java-tfidf5 ndcg@10: 0.4598704998596104
- java-dirichlet5 ndcg@10: 0.3520072908102156
- -----------------
- python-bm25 ndcg@10: 0.4968417431139021
- python-tfidf5 ndcg@10: 0.4680931950762395
- python-dirichlet5 ndcg@10: 0.3543755085621779
- -----------------
- javascript-bm25 ndcg@10: 0.5001025734259281
- javascript-tfidf5 ndcg@10: 0.47591826228966466
- javascript-dirichlet5 ndcg@10: 0.32928802463538187
- -----------------

